<a href="https://colab.research.google.com/github/Elwing-Chou/tibame20260212/blob/main/tibame20260307.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 象棋棋盤: https://i-blog.csdnimg.cn/blog_migrate/ee144a254459ba034fae2ff541b13d64.png
import pygame as pg

#pygame初始化
pg.init()

# --- 字型 ---
def get_font(size):
    for f in ['microsoftjhenghei', 'simhei', 'stheitirelight']:
        if f in pg.font.get_fonts():
            return pg.font.SysFont(f, size)
    return pg.font.SysFont(None, size)

# 32是字體的size
FONT_UI = get_font(32)

#設定視窗
width, height = 800, 880
inter = 80
# 產生視窗
screen = pg.display.set_mode([width, height])
# 設定遊戲標題
pg.display.set_caption("象棋")

# 準備我的轉換
role_dic = {
    # 腳色代號:(黑陣營, 紅陣營)
    0:("將", "帥"),
    1:("士", "仕"),
    2:("象", "相"),
    5:("包", "炮")
}

# 棋盤落子
row, col = 10, 9
board_role = [[-1] * col for i in range(row)]
# 真的有落子 (腳色, 陣營) 腳色 0:將/帥  陣營: 3(黑)/4(紅)
# 將帥
board_role[0][4] = (0, 0)
board_role[9][4] = (0, 1)
# 仕
board_role[0][3] = (1, 0)
board_role[0][5] = (1, 0)
board_role[9][3] = (1, 1)
board_role[9][5] = (1, 1)
# 像
board_role[0][2] = (2, 0)
board_role[0][6] = (2, 0)
board_role[9][2] = (2, 1)
board_role[9][6] = (2, 1)
# 包
board_role[2][1] = (5, 0)
board_role[2][7] = (5, 0)
board_role[7][1] = (5, 1)
board_role[7][7] = (5, 1)

# 把棋盤的座標預先存起來, 比較方便
row, col = 10, 9
board_cord = [[-1] * col for i in range(row)]
for i in range(row):
    for j in range(col):
        x = 80 * j + 80
        y = 80 * i + 80
        board_cord[i][j] = (x, y)

def draw():
    # 建立畫布bg
    bg = pg.Surface(screen.get_size())
    # 把畫布填滿某個顏色
    bg.fill([199, 167, 82])

    # 畫上半部棋盤
    # line(畫布, 顏色, 開始, 結束, 線寬度)
    # 畫橫線
    for i in range(5):
        pg.draw.line(bg,
                     [0, 0, 0],
                     [inter, inter*i+inter],
                     [9*inter, inter*i+inter],
                     2)
    # 畫直線
    for i in range(9):
        pg.draw.line(bg,
                     [0, 0, 0],
                     [inter*i+inter, inter],
                     [inter*i+inter, 5*inter],
                     2)
    # 畫兩條斜線
    pg.draw.line(bg,
                 [0, 0, 0],
                 [4*inter, inter],
                 [6*inter, 3*inter],
                 2)

    pg.draw.line(bg,
                 [0, 0, 0],
                 [6*inter, inter],
                 [4*inter, 3*inter],
                 2)


    # 畫下半部棋盤
    # 畫橫線
    for i in range(5):
        pg.draw.line(bg,
                     [0, 0, 0],
                     [inter, inter*i+6*inter],
                     [9*inter, inter*i+6*inter],
                     2)
    # 畫直線
    for i in range(9):
        pg.draw.line(bg,
                     [0, 0, 0],
                     [inter*i+inter, 6*inter],
                     [inter*i+inter, 10*inter],
                     2)
    # 畫兩條斜線
    pg.draw.line(bg,
                 [0, 0, 0],
                 [4*inter, 8*inter],
                 [6*inter, 10*inter],
                 2)

    pg.draw.line(bg,
                 [0, 0, 0],
                 [6*inter, 8*inter],
                 [4*inter, 10*inter],
                 2)

    # 畫出旗子
    # 塗層, 顏色, 中心, 半徑, 寬度(不給就是填滿)
    for i in range(row):
        for j in range(col):
            # 有落子
            if not board_role[i][j] == -1:
                x, y = board_cord[i][j]
                pg.draw.circle(bg,
                               (255, 255, 255),
                               (x, y),
                               30)
                # 腳色, 陣營
                ch, side = board_role[i][j]
                # 紅黑陣營的顏色
                if side == 0:
                    color = (0, 0, 0)
                else:
                    color = (255, 0, 0)
                # 準備填上去的字
                t = FONT_UI.render(role_dic[ch][side], 1, color)
                # 背景(bg)上面疊上t
                bg.blit(t, t.get_rect(center=(x, y)))

    # 你要把圖層放到上一層
    screen.blit(bg, [0, 0])
    # 對畫面進行更新(才會真的秀出來)
    pg.display.update()

# 再遊戲開始前,我們第一次繪製
draw()
# 建立一個永不結束的迴圈(遊戲才不會結束)
running = True
while running:
    # 收取你的遊戲任何事件(滑鼠點擊/鍵盤按鈕...)
    for event in pg.event.get():
        # 偵測滑鼠點擊以後放掉的動作
        if event.type == pg.MOUSEBUTTONUP:
            # 得到點擊時候的x, y座標
            x, y = pg.mouse.get_pos()
            print(x, y)
            mind, minposi, minposj = float("inf"), -1, -1
            # 我只要發現比mind還小的距離, 我就更新
            for i in range(row):
                for j in range(col):
                    cx, cy = board_cord[i][j]
                    d = abs(x-cx) + abs(y-cy)
                    if d < mind:
                        mind, minposi, minposj = d, i, j
            print(mind, board_cord[minposi][minposj])
        # 如果收到的事件是按x
        if event.type == pg.QUIT:
            # 迴圈就會變成while False
            running = False

pg.quit()